# G02: Your First Look Inside GPT-2

**Prerequisites:** G01 (conceptual understanding of mechinterp).

**What you'll learn:**
- How to load a pretrained transformer with IRTK
- What the model outputs and how to read its predictions
- How to capture every intermediate activation with `run_with_cache`
- The logit lens: watching predictions form layer by layer
- Residual stream norms and what they tell us

This is where you start getting your hands dirty. By the end, you'll be able to look inside any transformer and see what it's "thinking" at every layer.

---

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.logit_lens import logit_lens, logit_lens_top_k, logit_lens_correct_prob
from irtk.residual_stream import residual_norm_by_layer

print("Loading GPT-2 Small...")
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Loaded: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads/layer, "
      f"d_model={model.cfg.d_model}, vocab={model.cfg.d_vocab}")

## What the Model Sees

The model doesn't see words — it sees **token IDs**. GPT-2 uses byte-pair encoding (BPE), which splits text into subword pieces. Common words stay intact, but rare words get broken into smaller chunks. Understanding this is important because the model processes and predicts at the token level, not the word level.

In [ ]:
text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)
token_strs = [tokenizer.decode([t]) for t in np.array(tokens)]

print(f"Input text: {text!r}")
print(f"Number of tokens: {len(tokens)}")
print(f"\nToken ID -> String:")
for i, (tid, tstr) in enumerate(zip(np.array(tokens), token_strs)):
    print(f"  [{i}] {tid:>6d} -> {tstr!r}")

# Notice how some words become multiple tokens
print("\n--- Another example with unusual words ---")
text2 = "Tokenization is surprisingly counterintuitive!"
tokens2 = model.to_tokens(text2)
token_strs2 = [tokenizer.decode([t]) for t in np.array(tokens2)]
print(f"Input: {text2!r}")
print(f"Tokens: {token_strs2}")

## What the Model Outputs

The model outputs **logits** — a score for every token in the vocabulary at every position. At position *i*, the logits represent the model's prediction for what token comes at position *i+1*. Higher logits mean the model thinks that token is more likely. We convert logits to probabilities with softmax.

In [ ]:
text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)
token_strs = [tokenizer.decode([t]) for t in np.array(tokens)]

# Run the model
logits = model(tokens)
print(f"Logits shape: {logits.shape}  (seq_len={len(tokens)}, vocab_size={model.cfg.d_vocab})")

# Look at the last position — what does the model predict comes next?
last_logits = logits[-1]
probs = jax.nn.softmax(last_logits)

# Top 5 predictions
top5_indices = jnp.argsort(last_logits)[-5:][::-1]
print(f"\nTop 5 predictions for what comes after '{text}':")
for idx in top5_indices:
    idx = int(idx)
    print(f"  {tokenizer.decode([idx])!r:>12s}  (prob={float(probs[idx]):.4f})")

## Looking Inside — The Cache

So far we've only seen the model's final output. But the magic of interpretability is looking at **intermediate computations**. The `run_with_cache` method runs the model and records every internal activation — residual streams, attention patterns, MLP outputs, everything.

> This is the key to interpretability: instead of treating the model as a black box, we crack it open and examine every step of the computation.

In [ ]:
text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)
token_strs = [tokenizer.decode([t]) for t in np.array(tokens)]

# Run with cache to capture all intermediate activations
logits, cache = model.run_with_cache(tokens)

# What's inside the cache?
all_keys = sorted(cache.cache_dict.keys())
print(f"Cache contains {len(all_keys)} activations.\n")
print("A few notable ones:")

notable_keys = [
    "hook_embed",
    "hook_pos_embed",
    "blocks.0.hook_resid_pre",
    "blocks.0.attn.hook_pattern",
    "blocks.0.attn.hook_z",
    "blocks.0.hook_mlp_out",
    "blocks.0.hook_resid_post",
    "blocks.11.hook_resid_post",
]
for key in notable_keys:
    if key in cache.cache_dict:
        shape = cache.cache_dict[key].shape
        print(f"  {key:>40s}: shape={shape}")

print(f"\n--- What these shapes mean ---")
print(f"  Residual stream: [seq_len={len(tokens)}, d_model={model.cfg.d_model}]")
print(f"  Attention patterns: [n_heads={model.cfg.n_heads}, seq_len, seq_len]")
print(f"  Attention output (z): [seq_len, n_heads={model.cfg.n_heads}, d_head={model.cfg.d_head}]")

## The Logit Lens

Here's the key insight: the residual stream at every layer lives in the **same vector space** as the final output. That means we can apply the unembedding matrix to any intermediate layer and see what the model would predict *if computation stopped right there*.

This is the **logit lens** (nostalgebraist, 2020). It lets us watch the model's predictions form layer by layer — like watching a photograph develop in a darkroom.

In [ ]:
# Use a prompt where we know the answer
text = "The capital of France is"
tokens = model.to_tokens(text)
token_strs = [tokenizer.decode([t]) for t in np.array(tokens)]

# Get top-k predictions at every layer and position
top_k_results = logit_lens_top_k(model, tokens, k=5)

# Show how predictions evolve at the last position (predicting what comes after "is")
print(f"Prompt: {text!r}")
print(f"\nTop prediction at LAST position, layer by layer:")
print("=" * 65)

layer_names = ["embed"] + [f"layer {i:2d}" for i in range(model.cfg.n_layers)]
for i, name in enumerate(layer_names):
    preds = top_k_results[i][-1]  # last position
    top_token = tokenizer.decode([preds[0][0]])
    top_prob = preds[0][1]
    top3 = [f"{tokenizer.decode([p[0]])!r}({p[1]:.2f})" for p in preds[:3]]
    print(f"  {name}: {top_token!r:>10s} ({top_prob:.3f})  |  top-3: {', '.join(top3)}")

## Visualizing Prediction Convergence

Let's visualize this more systematically. For every position and every layer, we'll compute the probability that the model assigns to the *correct* next token. This gives us a heatmap of "when does the model figure out each token?"

In [ ]:
text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)
token_strs = [tokenizer.decode([t]) for t in np.array(tokens)]

# Get P(correct next token) at every layer and position
correct_probs = logit_lens_correct_prob(model, tokens)
# Shape: [n_layers+1, seq_len-1]

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(correct_probs, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

# Labels
y_labels = ["embed"] + [f"L{i}" for i in range(model.cfg.n_layers)]
x_labels = [f"{token_strs[j]}\n->{token_strs[j+1]}" for j in range(len(tokens)-1)]

ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels(y_labels)
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=45, ha='right', fontsize=8)
ax.set_xlabel("Position (current token -> correct next token)")
ax.set_ylabel("Layer")
ax.set_title("Logit Lens: P(correct next token) by Layer and Position")
plt.colorbar(im, ax=ax, label="Probability")
plt.tight_layout()
plt.show()

print("Bright = the model already 'knows' the answer at this layer.")
print("Dark = the model hasn't figured it out yet.")

## Comparing Multiple Prompts

Different types of knowledge converge at different layers. Syntactic patterns (like finishing a common phrase) tend to emerge early, while factual knowledge (like knowing a capital city) often requires deeper processing. Let's compare four prompts to see this in action.

In [ ]:
prompts = {
    "Factual": "The capital of France is",
    "Syntactic": "I went to the store and bought a bag of",
    "Cultural": "To be or not to be, that is the",
    "Arithmetic": "2 + 2 =",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
layer_labels = ["emb"] + [f"L{i}" for i in range(model.cfg.n_layers)]

for (label, prompt), ax in zip(prompts.items(), axes.flat):
    toks = model.to_tokens(prompt)

    # Get the model's actual prediction
    final_logits = model(toks)
    pred = int(jnp.argmax(final_logits[-1]))
    pred_str = tokenizer.decode([pred])

    # Get logit lens probabilities for the predicted token at the last position
    probs = logit_lens(model, toks, return_probs=True)
    pred_probs_by_layer = probs[:, -1, pred]

    ax.plot(pred_probs_by_layer, marker='o', markersize=3, color='#d62728')
    ax.fill_between(range(len(pred_probs_by_layer)), pred_probs_by_layer, alpha=0.15, color='#d62728')
    ax.set_title(f'{label}: "{prompt}" ->{pred_str!r}', fontsize=10)
    ax.set_ylabel("P(predicted token)")
    ax.set_ylim(-0.05, 1.05)
    ax.set_xticks(range(0, len(layer_labels), 2))
    ax.set_xticklabels(layer_labels[::2], fontsize=8)
    ax.set_xlabel("Layer")
    ax.grid(True, alpha=0.3)

plt.suptitle("Logit Lens: When Does the Model Decide?", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Residual Stream Norms

One more simple but revealing measurement: the **norm** (magnitude) of the residual stream at each layer. Since each layer adds its output to the residual stream, we'd expect the norm to grow. But *how* it grows tells us something about how the model allocates its computation.

In [ ]:
text = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(text)

# Run with cache so we can access intermediate residuals
logits, cache = model.run_with_cache(tokens)

# Get residual stream norms (averaged across positions)
norms = residual_norm_by_layer(cache)

layer_labels = ["embed"] + [f"L{i}" for i in range(model.cfg.n_layers)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(norms, marker='o', color='#1f77b4', linewidth=2)
ax.fill_between(range(len(norms)), norms, alpha=0.15, color='#1f77b4')
ax.set_xticks(range(len(layer_labels)))
ax.set_xticklabels(layer_labels, rotation=45, fontsize=9)
ax.set_xlabel("Layer")
ax.set_ylabel("L2 Norm (mean across positions)")
ax.set_title("Residual Stream Norm by Layer")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Norm at embedding: {norms[0]:.1f}")
print(f"Norm at final layer: {norms[-1]:.1f}")
print(f"Growth factor: {norms[-1] / norms[0]:.1f}x")

## Key Takeaways

1. **`from_pretrained` loads real model weights** from HuggingFace — you're working with the actual GPT-2 that was trained by OpenAI.
2. **`run_with_cache` captures every intermediate activation** — residual streams, attention patterns, MLP outputs, all of it.
3. **The logit lens shows predictions forming layer by layer** — you can literally watch the model "make up its mind."
4. **Different types of knowledge appear at different depths** — syntactic patterns converge early, factual recall often needs the later layers.
5. **Residual stream norms grow as the model adds information** — each layer contributes to a growing representation.

---

## Exercises

Try these on your own to build intuition:

1. **Your own factual prompts.** Run the logit lens on prompts like `"The largest planet in the solar system is"` or `"The CEO of Tesla is"`. At which layer does the correct answer emerge?

2. **When the model is wrong.** Find a prompt where GPT-2 predicts the wrong next token. Use the logit lens to see when the wrong answer first appears — was it always wrong, or did it start correct and get overwritten?

3. **Different positions.** The logit lens heatmap shows all positions. Pick a sentence and compare: which positions are "easy" (converge early) and which are "hard" (need more layers)? Why might that be?

4. **Residual norms for different inputs.** Compare residual stream norms for a short vs. long sentence, or for a factual vs. nonsensical prompt. Does the growth pattern change?

## Further Reading

- **nostalgebraist (2020)**, ["interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — the original blog post introducing the logit lens.
- **Belrose et al. (2023)**, ["Eliciting Latent Predictions from Transformers with the Tuned Lens"](https://arxiv.org/abs/2303.08112) — the tuned lens, which learns per-layer probes for more accurate intermediate predictions.
- **Next up: G03** — we'll zoom in on attention heads and start understanding *how* the model moves information between tokens.